In [12]:
import xml.etree.ElementTree as ET
import os
import cv2
import shutil

In [20]:
def convert_bbox(size, box):
    # size: (img_width, img_height)
    # box: (xmin, ymin, xmax, ymax)
    # return: (x_center, y_center, width, height) 0~1 값

    img_width = size[0]
    img_height = size[1]

    xmin = box[0]
    ymin = box[1]
    xmax = box[2]
    ymax = box[3]
    
    dw = 1.0 / img_width
    dh = 1.0 / img_height

    x_center = dw * (xmin + xmax) / 2
    y_center = dh * (ymin + ymax) / 2
    w = (xmax - xmin) * dw
    h = (ymax - ymin) * dh

    # 안전장치: 0.0 ~ 1.0 범위를 벗어나지 않도록 제한 (Clipping)
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    w = max(0.0, min(1.0, w))
    h = max(0.0, min(1.0, h))

    return (x_center, y_center, w, h)

def parse_xml_to_txt(image_and_xml_dir, pure_name, ext):
    class_mapping = {"apple": 0, "banana": 1, "orange": 2}

    xml_path = os.path.join(image_and_xml_dir, f"{pure_name}.xml")

    if not os.path.exists(xml_path):
        return []
    
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # 이미지 크기 추출
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    # width/height가 0이면 실제 이미지 파일에서 크기를 읽어온다.
    if w == 0 or h == 0:
        filename = root.find('filename').text
        image_path = os.path.join(image_and_xml_dir, f"{pure_name}{ext}")

        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            if img is not None:
                h, w, _ = img.shape
            else:
                return [] # 이미지 파일이 깨졌다면 이 파일은 건너뜀
        else:
            return [] # 이미지 파일이 없으면 건너뜀
    

    yolo_labels = []

    # 객체 태그 반복문
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        if class_name not in class_mapping:
            continue

        class_id = class_mapping[class_name]
        xmlbox = obj.find('bndbox')

        b = (float(xmlbox.find('xmin').text), 
             float(xmlbox.find('ymin').text), 
             float(xmlbox.find('xmax').text), 
             float(xmlbox.find('ymax').text))

        # 좌표 변환 호출
        bb = convert_bbox((w, h), b)

        # YOLO 포맷: "class_id x_center y_center width, height"
        yolo_labels.append(f"{class_id} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}")

    if not yolo_labels:
        return []

    save_path = "dataset/fruits/fruit-images-for-object-detection/train_zip"

    # 이미지 복사
    image_src_path = os.path.join(image_and_xml_dir, f"{pure_name}{ext}")
    image_dest_path = os.path.join(save_path, f"images/{pure_name}{ext}")
    shutil.copy(image_src_path, image_dest_path)

    # txt 라벨 파일 저장 (join 메소드로 개행 깔끔히 처리)
    txt_save_path = os.path.join(save_path, f"labels/{pure_name}.txt")
    
    with open(txt_save_path, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_labels))
                
    return yolo_labels

In [7]:
xml_path = "dataset/fruits/fruit-images-for-object-detection/train_zip/train/mixed_1.xml"
image_dir = "dataset/fruits/fruit-images-for-object-detection/train_zip/train"
class_mapping = {"apple": 0, "banana": 1, "orange": 2}
yolo_labels = parse_xml_to_txt(xml_path, image_dir, class_mapping)
print(yolo_labels)

['2 0.427500 0.373227 0.302500 0.452128', '0 0.691250 0.489362 0.282500 0.453901', '1 0.425625 0.670213 0.716250 0.531915']


```
일단.. 캐글에서 xml이 있는 객체 탐지 데이터셋을 다운받았는데, 한 폴더 안에 xml, image가 있었다.
다른 데이터셋도 그런진 모르겠지만, 일단 이 데이터셋 기준으로 함수를 재구성 해보자.
xml 파일을 파싱해서 yolo_labels를 만들고, 만든 것으로 txt 파일에 writeline을 해서 내가 새로 만든 labels 폴더에 넣을거다.
일단 해보자.. just do it
```

In [27]:
common_path = "dataset/fruits/fruit-images-for-object-detection"
# image_and_xml_dir = "dataset/fruits/fruit-images-for-object-detection/train_zip/train"

split_maps = {"train": {"image_and_xml_dir": os.path.join(common_path, "train_zip/train")}, 
             "valid": {"image_and_xml_dir": os.path.join(common_path, "test_zip/test")}}

for split_name, info in split_maps.items():
    images = [f for f in os.listdir(info["image_and_xml_dir"]) if f.endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    print(len(images))
    print(split_name)
    break
    for image in images:
        pure_name, ext = os.path.splitext(image)
        parse_xml_to_txt(image_and_xml_dir, pure_name, ext)




240
train


In [22]:
images_and_xml_dir = "dataset/fruits/fruit-images-for-object-detection/test_zip/test"
images = [f for f in os.listdir(image_and_xml_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.webp'))]

for image in images:
    pure_name, ext = os.path.splitext(image)
    parse_xml_to_txt(image_and_xml_dir, pure_name, ext)

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
